# CS 363M — Wildlife strike: data cleaning and feature engineering

**Authors:** (your names)

This notebook:
- Summarizes **missing data** and **target** distribution
- **Explores** time patterns (year, month, hour) and a lat/lon sample
- **Cleans** data: drop ultra-sparse columns (≥ 80% missing) and free text; **exclude leakage**; **impute** (median for numeric, mode / `"Unknown"` for categorical)
- **Engineers** features: one calendar basis (`YEAR`, `MONTH`, `QUARTER`, `DOY`, `HOUR_IMPUTED`), **frequency** encodings, **one-hot** for medium-cardinality fields, **standardizes** `HEIGHT`, `SPEED`, `DISTANCE`, `HOUR_IMPUTED`
- Produces a numeric matrix `X` and `y` ready for e.g. `sklearn.ensemble.GradientBoostingClassifier` and nested CV (next step)

In [39]:
# Standard Headers
# You may add additional headers here if needed
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

# Enable inline mode for matplotlib so that Jupyter displays graphs
%matplotlib inline

# print your pandas version
pd.__version__ 

'3.0.2'

## 1. Load and target

The competition target is `INDICATED_DAMAGE` (0/1).

In [40]:
data = pd.read_csv("train.csv")

data.head()

/var/folders/r4/ctd1pj6s2hs6t2wbmr69730w0000gn/T/ipykernel_15549/2074409587.py:1: DtypeWarning: Columns (0: LATITUDE, 1: LONGITUDE, 2: AMO) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("train.csv")


,INDEX_NR,INCIDENT_DATE,INCIDENT_MONTH,INCIDENT_YEAR,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,LATITUDE,LONGITUDE,...,NUM_SEEN,NUM_STRUCK,SIZE,ENROUTE_STATE,COMMENTS,SOURCE,PERSON,LUPDATE,TRANSFER,INDICATED_DAMAGE
0,1410120,12/13/93,12,1993,NaN,Day,TJSJ,LUIS MUNOZ MARIN INTL,18.43942,-66.00183,...,10-Feb,10-Feb,Small,NaN,NaN,FAA Form 5200-7,Pilot,4/3/23,0,0
1,709688,2/1/10,2,2010,5:00,Night,WMKK,KUALA LUMPUR INTL,2.745578,101.709917,...,NaN,1,Medium,NaN,2010-5-18-53374 /Legacy Record 300758/,FAA Form 5200-7-E,Air Transport Operations,6/9/10,0,0
2,730841,5/9/12,5,2012,2:00,Night,KSDF,MUHAMMAD ALI INTERNATIONAL,38.17439,-85.736,...,NaN,1,Large,NaN,UPS EVENT REPT 36216 (4/22/13 UPDATED COST) /L...,Air Transport Report,Air Transport Operations,4/22/13,0,1
3,654676,10/8/02,10,2002,NaN,NaN,KLAX,LOS ANGELES INTL,33.94254,-118.40807,...,NaN,10-Feb,Medium,NaN,2002-10-8-111929 /Legacy Record 216397/,FAA Form 5200-7-E,Carcass Found,1/9/03,0,0
4,629708,2/3/97,2,1997,NaN,Dawn,PHLI,LIHUE ARPT,21.97598,-159.33896,...,1,1,Medium,NaN,SOURCE 5200-7 & PACIR /Legacy Record 121531/,Multiple,NaN,3/1/07,0,0


## 2. Drop not useful data
REMARKS, NUM_STRUCK, LOCATION, ENG_3_POS, BIRD_BAND_NUMBER, ENG_4_POS, ENROUTE_STATE, PRECIPITATION, COMMONS, TRANSFER, SOURCE, LUPDATE


In [41]:
cols_to_drop = [
    "REMARKS",
    "NUM_STRUCK",
    "LOCATION",
    "ENG_3_POS",
    "BIRD_BAND_NUMBER",
    "ENG_4_POS",
    "ENROUTE_STATE",
    "PRECIPITATION", # TRY THIS WITH AND WITHOUT THIS COLUMN
    "COMMENTS",
    "TRANSFER",
    "SOURCE",
    "LUPDATE", # last updated time
    "RUNWAY", # runway number
    "FLT", # flight number
    "AIRCRAFT", # name of the aircraft in text
    "INDEX_NR", # index number
    "REG", 
    "PERSON",
    "NUM_SEEN",
    "NUM_STRUCK"

]


df = data.drop(columns=cols_to_drop, errors="ignore")

df.head()

,INCIDENT_DATE,INCIDENT_MONTH,INCIDENT_YEAR,TIME,TIME_OF_DAY,AIRPORT_ID,AIRPORT,LATITUDE,LONGITUDE,STATE,...,DISTANCE,SKY,SPECIES_ID,SPECIES,OUT_OF_RANGE_SPECIES,REMAINS_COLLECTED,REMAINS_SENT,WARNED,SIZE,INDICATED_DAMAGE
0,12/13/93,12,1993,NaN,Day,TJSJ,LUIS MUNOZ MARIN INTL,18.43942,-66.00183,PR,...,NaN,Some Cloud,UNKBS,Unknown bird - small,0,1,0,No,Small,0
1,2/1/10,2,2010,5:00,Night,WMKK,KUALA LUMPUR INTL,2.745578,101.709917,FN,...,0.0,NaN,UNKBM,Unknown bird - medium,0,0,0,Unknown,Medium,0
2,5/9/12,5,2012,2:00,Night,KSDF,MUHAMMAD ALI INTERNATIONAL,38.17439,-85.736,KY,...,8.0,NaN,UNKBL,Unknown bird - large,0,0,0,No,Large,1
3,10/8/02,10,2002,NaN,NaN,KLAX,LOS ANGELES INTL,33.94254,-118.40807,CA,...,0.0,NaN,NE120,Western gull,0,1,0,Unknown,Medium,0
4,2/3/97,2,1997,NaN,Dawn,PHLI,LIHUE ARPT,21.97598,-159.33896,HI,...,0.0,Some Cloud,R1101,American barn owl,0,0,0,No,Medium,0


## Merge the columns that are redudant

In [42]:
# converting date to a contionous variable and then dropping the year keeping the month
d = pd.to_datetime(df["INCIDENT_DATE"], format="mixed", dayfirst=False, errors="coerce").dt.normalize()
anchor = d.min()
df["INCIDENT_DATE"] = (d - anchor).dt.days

df = df.drop(columns=["INCIDENT_YEAR"], errors="ignore")


p = df["TIME"].astype(str).str.split(":", n=1, expand=True)
h = pd.to_numeric(p[0], errors="coerce")
if 1 in p.columns:
    m = pd.to_numeric(p[1], errors="coerce").fillna(0)
else:
    m = pd.Series(0, index=df.index)

df["TIME"] = (h * 60 + m).where(
    h.between(0, 23) & m.between(0, 59)
)

df = df.drop(columns=["TIME_OF_DAY"], errors="ignore")

# AIRPORT_ID, OPID, SPECIES_ID as the replacement for redundant columns
df = df.drop(columns=["AIRPORT"], errors="ignore")
df = df.drop(columns=["OPERATOR"], errors="ignore")
df = df.drop(columns=["SPECIES"], errors="ignore")

In [43]:
df.head()

,INCIDENT_DATE,INCIDENT_MONTH,TIME,AIRPORT_ID,LATITUDE,LONGITUDE,STATE,FAAREGION,OPID,AMA,...,SPEED,DISTANCE,SKY,SPECIES_ID,OUT_OF_RANGE_SPECIES,REMAINS_COLLECTED,REMAINS_SENT,WARNED,SIZE,INDICATED_DAMAGE
0,1441,12,NaN,TJSJ,18.43942,-66.00183,PR,ASO,AAL,148,...,145.0,NaN,Some Cloud,UNKBS,0,1,0,No,Small,0
1,7335,2,300.0,WMKK,2.745578,101.709917,FN,FGN,FDX,583,...,NaN,0.0,NaN,UNKBM,0,0,0,Unknown,Medium,0
2,8163,5,120.0,KSDF,38.17439,-85.736,KY,ASO,UPS,04A,...,240.0,8.0,NaN,UNKBL,0,0,0,No,Large,1
3,4662,10,NaN,KLAX,33.94254,-118.40807,CA,AWP,UNK,NaN,...,NaN,0.0,NaN,NE120,0,1,0,Unknown,Medium,0
4,2589,2,NaN,PHLI,21.97598,-159.33896,HI,AWP,1AAH,148,...,135.0,0.0,Some Cloud,R1101,0,0,0,No,Medium,0


## Impude the data

In [44]:
# numeric
for c in df.select_dtypes(include=[np.number]).columns:
    # df[c] = df[c].fillna(df[c].median())
    df[c] = df[c].fillna(-1)

# categorical → explicit missing bucket
for c in df.select_dtypes(include=["str", "object", "category"]).columns:
    df[c] = df[c].fillna("Unknown")

## encode the data

In [45]:
# y = target; all other columns = features
TARGET = "INDICATED_DAMAGE"
y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET], errors="ignore")

# integer codes for all text / object columns (fit on train only)
text_cols = list(X.select_dtypes(include=["str", "object", "string", "category"]).columns)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_test = X_train.copy(), X_test.copy()

# OrdinalEncoder needs one comparable type per column (CSV "object" can mix float + str)
def _as_cat_str(ser: pd.Series) -> pd.Series:
    s = ser.map(lambda v: "Unknown" if (v is None or (isinstance(v, float) and np.isnan(v)) or pd.isna(v)) else v)
    s = s.astype("string[python]").astype(str)  # uniform strings
    return s.replace({"nan": "Unknown", "None": "Unknown", "<NA>": "Unknown", "": "Unknown"})

if text_cols:
    for c in text_cols:
        X_train[c] = _as_cat_str(X_train[c])
        X_test[c] = _as_cat_str(X_test[c])

    enc = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        encoded_missing_value=-1,
        dtype=np.int64,
    )
    enc.fit(X_train[text_cols])
    X_train[text_cols] = enc.transform(X_train[text_cols]).astype(np.int64)
    X_test[text_cols] = enc.transform(X_test[text_cols]).astype(np.int64)
else:
    enc = None

# any leftover object / str -> numeric, then float matrix for sklearn
for c in X_train.columns:
    if X_train[c].dtype == "object" or str(X_train[c].dtype) == "str":
        X_train[c] = pd.to_numeric(X_train[c], errors="coerce")
        X_test[c] = pd.to_numeric(X_test[c], errors="coerce")
        m = X_train[c].median()
        X_train[c] = X_train[c].fillna(m)
        X_test[c] = X_test[c].fillna(m)

X_train = X_train.astype(float)
X_test = X_test.astype(float)

print("X_train", X_train.shape, "| X_test", X_test.shape, "| label-encoded:", text_cols)

X_train (245742, 30) | X_test (61436, 30) | label-encoded: ['AIRPORT_ID', 'LATITUDE', 'LONGITUDE', 'STATE', 'FAAREGION', 'OPID', 'AMA', 'AMO', 'AC_CLASS', 'TYPE_ENG', 'PHASE_OF_FLIGHT', 'SKY', 'SPECIES_ID', 'WARNED', 'SIZE']


In [60]:
X.head(10)

,INCIDENT_DATE,INCIDENT_MONTH,TIME,AIRPORT_ID,LATITUDE,LONGITUDE,STATE,FAAREGION,OPID,AMA,...,HEIGHT,SPEED,DISTANCE,SKY,SPECIES_ID,OUT_OF_RANGE_SPECIES,REMAINS_COLLECTED,REMAINS_SENT,WARNED,SIZE
0,1441,12,-1.0,TJSJ,18.43942,-66.00183,PR,ASO,AAL,148,...,300.0,145.0,-1.0,Some Cloud,UNKBS,0,1,0,No,Small
1,7335,2,300.0,WMKK,2.745578,101.709917,FN,FGN,FDX,583,...,50.0,-1.0,0.0,Unknown,UNKBM,0,0,0,Unknown,Medium
2,8163,5,120.0,KSDF,38.17439,-85.736,KY,ASO,UPS,04A,...,3500.0,240.0,8.0,Unknown,UNKBL,0,0,0,No,Large
3,4662,10,-1.0,KLAX,33.94254,-118.40807,CA,AWP,UNK,Unknown,...,-1.0,-1.0,0.0,Unknown,NE120,0,1,0,Unknown,Medium
4,2589,2,-1.0,PHLI,21.97598,-159.33896,HI,AWP,1AAH,148,...,0.0,135.0,0.0,Some Cloud,R1101,0,0,0,No,Medium
5,11165,7,538.0,RCTP,25.077731,121.232822,FN,FGN,FDX,148,...,0.0,110.0,0.0,No Cloud,UNKBS,0,0,0,Unknown,Small
6,7474,6,432.0,KMSY,29.99339,-90.25803,LA,ASW,JBU,04A,...,0.0,-1.0,0.0,No Cloud,UNKBS,0,0,0,Yes,Small
7,6074,8,460.0,KSBN,41.70895,-86.31847,IN,AGL,FLG*,188,...,0.0,-1.0,0.0,No Cloud,O2205,0,1,0,No,Small
8,4628,9,-1.0,KJFK,40.63975,-73.77893,NY,AEA,AAL,148,...,0.0,-1.0,0.0,Unknown,K2001,0,1,0,Unknown,Large
9,10154,10,1238.0,KEWR,40.6925,-74.16866,NJ,AEA,UAL,04A,...,0.0,-1.0,0.0,No Cloud,R2203,0,1,0,Unknown,Large


Train a model

In [59]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# expects X_train, X_test, y_train, y_test from the encoding cell above

base = DecisionTreeClassifier(
    max_depth=15,           # stumps (default AdaBoost “weak learner”)
    min_samples_leaf=100,
    class_weight="balanced",  # helps if damage is rare; optional
    random_state=42,
)

clf = AdaBoostClassifier(
    estimator=base,
    n_estimators=300,      # more rounds = more fit capacity (watch overfit)
    learning_rate=0.1,    # 0.3–1.0 common; lower → need more stumps
    random_state=42,       # sklearn >= 1.4: no `algorithm=` kwarg (SAMME is default)
)

clf.fit(X_train, y_train)


p = clf.predict_proba(X_test)[:, 1]
print("ROC-AUC:", roc_auc_score(y_test, p))
print("PR-AUC:", average_precision_score(y_test, p))
print(classification_report(y_test, clf.predict(X_test), digits=4))

ROC-AUC: 0.9047264948356852
PR-AUC: 0.5061127459394116
              precision    recall  f1-score   support

           0     0.9808    0.8748    0.9248     57531
           1     0.2884    0.7475    0.4162      3905

    accuracy                         0.8667     61436
   macro avg     0.6346    0.8111    0.6705     61436
weighted avg     0.9368    0.8667    0.8924     61436



## create a submission file

In [61]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "scripts/make_submission.py",
        "--train", "train.csv",
        "--test", "test.csv",
        "--out", "submission.csv",
    ],
)
print("Wrote submission.csv")

/opt/homebrew/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/Resources/Python.app/Contents/MacOS/Python: can't open file '/Users/noorali/dev/ml-1-project/scripts/make_submission.py': [Errno 2] No such file or directory


CalledProcessError: Command '['/Users/noorali/dev/ml-1-project/.venv/bin/python', 'scripts/make_submission.py', '--train', 'train.csv', '--test', 'test.csv', '--out', 'submission.csv']' returned non-zero exit status 2.

/opt/homebrew/Cellar/python@3.13/3.13.7/Frameworks/Python.framework/Versions/3.13/Resources/Python.app/Contents/MacOS/Python: can't open file '/Users/noorali/dev/ml-1-project/scripts/make_submission.py': [Errno 2] No such file or directory


CalledProcessError: Command '['/Users/noorali/dev/ml-1-project/.venv/bin/python', 'scripts/make_submission.py', '--train', 'train.csv', '--test', 'test.csv', '--out', 'submission.csv']' returned non-zero exit status 2.